## Mamogram analysis

[Nationwide real-world implementation of AI for cancer detection in population-based mammography screening](https://www.nature.com/articles/s41591-024-03408-6)

Download the praim.csv from here.
https://datadryad.org/dataset/doi:10.5061/dryad.zs7h44jgn

In [1]:
import pandas as pd

mamogram_file = "~/Downloads/praim.csv"



In [2]:
import random
def pipe_filter_exclude_ai_viewer(df):

   return df.query("used_ai_viewer == False").copy()


def compute_pd(df):
   #  first and second read are the same no consense ( 3rd label is NOT used)
   df["pd"] = df[["first_read",	"second_read"]].apply(lambda x: 1.0 if x.iloc[0] == x.iloc[1] else round(2/3,2), axis=1)
   df["pd_majority"] = df["Majority label"]
   
   # Now change the majority label ("NOT Match" when no alignment)  to either sus or normal based on whether the patient was recalled.
   df.loc[df['pd_majority'] == 'NOT MATCH', "pd_majority"] = df.loc[df['pd_majority'] == 'NOT MATCH']["had_recall"].apply(lambda x: "not-normal" if x else "normal" )
   df.loc[df['pd_majority'] == 'suspicious', "pd_majority"] = "not-normal"

   df["all_humans"]= df.apply(lambda x: list([x["first_read"],	x["second_read"],  x['pd_majority']]) if x["Majority label"] == "NOT MATCH"  else  list([x["first_read"],	x["second_read"]]) , axis=1)

   for i in range(5):
        df[f"another_human_{i}"] = df.apply(lambda x: random.choice( x["all_humans"] ), axis=1)
        # Rename value from suspicious to not-normal
        df.loc[df[f"another_human_{i}"] == 'suspicious', f"another_human_{i}"] = "not-normal"



   return df
    
df = pd.read_csv(mamogram_file).pipe(pipe_filter_exclude_ai_viewer).pipe(compute_pd)



In [3]:
df.sample(n=10)

,study_id,cancer_detected,had_recall,used_ai_viewer,reader_set,readers,ai_prediction,had_consensus_conference,had_pre_operation_biopsy,screening_date,...,safety_net_shown,safety_net_accepted,pd,pd_majority,all_humans,another_human_0,another_human_1,another_human_2,another_human_3,another_human_4
326873,326874,False,False,False,rp268,7_101,normal,False,False,2022-H2,...,False,False,1.00,normal,"[normal, normal]",normal,normal,normal,normal,normal
431284,431285,False,False,False,rp308,35_47,not-normal,False,False,2022-H2,...,False,False,1.00,normal,"[normal, normal]",normal,normal,normal,normal,normal
120390,120391,False,False,False,rp527,20_43,not-normal,False,False,2021-H2,...,False,False,1.00,normal,"[normal, normal]",normal,normal,normal,normal,normal
44879,44880,False,False,False,rp287,63_82,not-normal,False,False,2022-H1,...,False,False,1.00,normal,"[normal, normal]",normal,normal,normal,normal,normal
228794,228795,False,False,False,rp234,6_37,normal,False,False,2022-H2,...,False,False,1.00,normal,"[normal, normal]",normal,normal,normal,normal,normal
379428,379429,False,False,False,rp228,78_86,not-normal,False,False,2021-H2,...,False,False,1.00,normal,"[normal, normal]",normal,normal,normal,normal,normal
386367,386368,False,False,False,rp524,43_88,normal,True,False,2022-H2,...,False,False,0.67,normal,"[normal, suspicious, normal]",normal,not-normal,normal,normal,not-normal
135666,135667,False,False,False,rp271,1_56,normal,False,False,2023-H1,...,False,False,1.00,normal,"[normal, normal]",normal,normal,normal,normal,normal
44030,44031,False,False,False,rp287,63_82,not-normal,False,False,2022-H1,...,False,False,1.00,normal,"[normal, normal]",normal,normal,normal,normal,normal
127688,127689,False,False,False,rp233,37_65,normal,False,False,2022-H2,...,False,False,1.00,normal,"[normal, normal]",normal,normal,normal,normal,normal


In [4]:
from sklearn.metrics import classification_report, accuracy_score


def report_by_pd(df, prediction_label = "ai_prediction", predictor_friendly_name="AI"):
    items = []
    scores_dict = classification_report(df["pd_majority"], df[prediction_label],  output_dict=True, zero_division=0.0)["not-normal"]
    scores_dict["PA"]     = "ALL"
    scores_dict["S"]=len(df)
    scores_dict["accuracy"] = accuracy_score(df["pd_majority"], df[prediction_label])

    items.append(scores_dict)
    
    st_groups = list(df["pd"].unique())

    for i in st_groups:



        d_subset  = df.query(f"pd ==  {i}")
        d = classification_report(d_subset["pd_majority"], d_subset[prediction_label],  output_dict=True, zero_division=0.0)      
        if "not-normal"  in d:
            d_treu = d["not-normal"]
            d_treu["PA"]= round(i, 2)
            d_treu["raw_PA"]= i
            d_treu["accuracy"] =accuracy_score(d_subset["pd_majority"], d_subset[prediction_label])
            d_treu["S"]= len(d_subset)
            items.append(d_treu)

    df_scores = pd.DataFrame(items)[["S","PA","accuracy", "precision", "recall", "f1-score", "support"]].round(2)
    df_scores["support"] = df_scores["support"].div(df_scores["S"]).apply(lambda x: "{:.3f}".format(x)) 

    df_scores = df_scores[["PA","S", "support", "f1-score", "precision", "recall", "accuracy"]]
    idx = pd.MultiIndex.from_tuples(
        [
         (predictor_friendly_name, c)  if c in ["accuracy", "precision", "recall", "f1-score"] else  ("common", c)   for c in df_scores.columns
        ],
        names=["predictor", "score_type"]
    )

    df_scores.columns= pd.MultiIndex.from_tuples(idx)
    return df_scores
    
            
df_scores_ai = report_by_pd(df)
df_scores_ai

common                       AI                          
      PA       S support f1-score precision recall accuracy
0    ALL  201079   0.045     0.15      0.08   0.85     0.56
1    1.0  185245   0.029     0.11      0.06   0.89     0.58
2   0.67   15834   0.229     0.39      0.26   0.79     0.42

In [5]:
from functools import reduce

another_human_scores = []
for l in [l for l in list(df.columns) if l.startswith("another_human")]:

     another_human_scores.append(report_by_pd(df, l, l))

# Merge all DataFrames on the 'key_col' column
df_scores_human_all = reduce(lambda left, right: pd.merge(left, right, on=[("common", "PA" ), ("common", "S" ), ("common", "support" )], how='inner'), another_human_scores)

df_scores_human_all

common                 another_human_0                            \
      PA       S support        f1-score precision recall accuracy   
0    ALL  201079   0.045            0.75      0.66   0.87     0.97   
1    1.0  185245   0.029            1.00      1.00   1.00     1.00   
2   0.67   15834   0.229            0.48      0.37   0.67     0.66   

  another_human_1                   ... another_human_2           \
         f1-score precision recall  ...          recall accuracy   
0            0.75      0.66   0.87  ...            0.87     0.97   
1            1.00      1.00   1.00  ...            1.00     1.00   
2            0.48      0.37   0.68  ...            0.67     0.67   

  another_human_3                           another_human_4                   \
         f1-score precision recall accuracy        f1-score precision recall   
0            0.75      0.66   0.87     0.97            0.75      0.66   0.87   
1            1.00      1.00   1.00     1.00            1.00      1.00   1.00   
2            0.48      0.38   0.67     0.67            0.48      0.37   0.67   

            
  accuracy  
0     0.97  
1     1.00  
2     0.67  

[3 rows x 23 columns]

In [6]:
df_scores_human_mean = df_scores_human_all.loc[:, pd.IndexSlice["common", :]].copy()
for c in ["recall", "precision", "f1-score", "accuracy"]:
    df_scores_human_mean.loc[:,("another_human", c)] = df_scores_human_all.loc[:, pd.IndexSlice[:, c]].mean(axis=1)
    df_scores_human_mean.loc[:,("another_human", f"{c}_std")] = df_scores_human_all.loc[:, pd.IndexSlice[:,c]].std(axis=1)
    df_scores_human_mean[("another_human", f"fmt_{c}_meanstd")] = df_scores_human_mean[[("another_human", c),("another_human", f"{c}_std")]].apply(lambda x: f"{x[0]:.2f}$\\pm${x[1]:.4f}",axis=1)

df_scores_human_mean

/var/folders/7v/5_mr86mx7l9g94fxzdpdx0nw0000gn/T/ipykernel_14483/2414675394.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_scores_human_mean[("another_human", f"fmt_{c}_meanstd")] = df_scores_human_mean[[("another_human", c),("another_human", f"{c}_std")]].apply(lambda x: f"{x[0]:.2f}$\\pm${x[1]:.4f}",axis=1)
/var/folders/7v/5_mr86mx7l9g94fxzdpdx0nw0000gn/T/ipykernel_14483/2414675394.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_scores_human_mean[("another_human", f"fmt_{c}_meanstd")] = df_scores_human_mean[[("another_human", c),("another_human", f"{c}_std")]].apply(lambda x: f"{x[0]:.2f}$\\pm${x[1]:

common                 another_human                                   \
      PA       S support        recall    recall_std fmt_recall_meanstd   
0    ALL  201079   0.045         0.870  1.241267e-16    0.87$\pm$0.0000   
1    1.0  185245   0.029         1.000  0.000000e+00    1.00$\pm$0.0000   
2   0.67   15834   0.229         0.672  4.472136e-03    0.67$\pm$0.0045   

                                                                       \
  precision precision_std fmt_precision_meanstd f1-score f1-score_std   
0     0.660      0.000000       0.66$\pm$0.0000     0.75          0.0   
1     1.000      0.000000       1.00$\pm$0.0000     1.00          0.0   
2     0.374      0.005477       0.37$\pm$0.0055     0.48          0.0   

                                                                   
  fmt_f1-score_meanstd accuracy accuracy_std fmt_accuracy_meanstd  
0      0.75$\pm$0.0000    0.970     0.000000      0.97$\pm$0.0000  
1      1.00$\pm$0.0000    1.000     0.000000      1.00$\pm$0.0000  
2      0.48$\pm$0.0000    0.668     0.004472      0.67$\pm$0.0045

In [7]:
index_columns = [("common", "PA" ), ("common", "S" ), ("common", "support" )]
df_scores_all = df_scores_human_mean.merge(df_scores_ai, on=index_columns)
df_scores_all = df_scores_all.set_index(index_columns)

expected_formula_map = {
    'f1-score': lambda pd, m: (2*m*pd)/(2*m*pd + 1-pd),
    "accuracy": lambda pd, m: pd,
    "recall": lambda pd, m: pd,
    "precision": lambda pd, m: (m*pd)/(m*pd + (1-m)* (1-pd)),
}

def expected_all_accuracy(listdict):
    num_sum = 0
    denom_sum = 0
    for t in listdict :
       if  t["p"] == "ALL": continue
       num_sum =  num_sum + t["s"]* t["p"]
       denom_sum=denom_sum+ t["s"]
    return num_sum/denom_sum

def expected_all_recall(listdict):
    num_sum = 0
    denom_sum = 0
    for t in listdict :
       if  t["p"] == "ALL": continue
       num_sum =  num_sum + t["s"]*t["p"]* t["m"]
       denom_sum=denom_sum+ t["s"]*t["m"]
    return num_sum/denom_sum

def expected_all_precision(listdict):
    num_sum = 0
    denom_sum = 0
    for t in listdict :

       if  t["p"] == "ALL": continue
       num_sum =  num_sum + t["s"]*t["p"]* t["m"]
       denom_sum = denom_sum + t["s"]*(t["p"]*t["m"]+ (1-t["p"])*(1-t["m"]))
    return num_sum/denom_sum

def expected_all_f1(listdict):

    numerator =  2* expected_all_precision(listdict)*expected_all_recall(listdict)
    denom =  expected_all_precision(listdict) + expected_all_recall(listdict)
    return numerator/denom

expected_all_formula_map = {
    'f1-score': expected_all_f1,
    "accuracy": expected_all_accuracy,
    "recall": expected_all_recall,
    "precision": expected_all_precision,
}

def extract_pd_s_m(df):
    return [{"p": x[0], "s": x[1], "m": float(x[2])}
            for x in df.index]

for s in ["f1-score", "precision", "recall", "accuracy" ]:
    df_scores_all[("delta_h_ai", s)] = df_scores_all[("another_human", s )]  - df_scores_all[("AI", s )]

for s in ["f1-score", "precision", "recall", "accuracy" ]:

    df_scores_all[("Expected", s)] = list(pd.DataFrame(list(df_scores_all.index)).apply(lambda x: expected_formula_map[s](float(x[0]), float(x[2])) if x[0] != 'ALL' else expected_all_formula_map[s](extract_pd_s_m(df_scores_all)), axis=1 ))

    # df_scores_all[("Expected", s)] = list(pd.DataFrame(list(df_scores_all.index)).apply(lambda x: expected_formula_map[s](float(x[0]), float(x[2])) if x[0] != 'ALL' else "-", axis=1 ))

    df_scores_all[("delta_e_h", s)] = df_scores_all[("Expected", s)]  - df_scores_all[("another_human", s )]

    df_scores_all[("delta_e_ai", s)] = df_scores_all[("Expected", s)]  - df_scores_all[("AI", s )]

df_scores_all

another_human                \
                                                  recall    recall_std   
(common, PA) (common, S) (common, support)                               
ALL          201079      0.045                     0.870  1.241267e-16   
1.0          185245      0.029                     1.000  0.000000e+00   
0.67         15834       0.229                     0.672  4.472136e-03   

                                                                         \
                                           fmt_recall_meanstd precision   
(common, PA) (common, S) (common, support)                                
ALL          201079      0.045                0.87$\pm$0.0000     0.660   
1.0          185245      0.029                1.00$\pm$0.0000     1.000   
0.67         15834       0.229                0.67$\pm$0.0045     0.374   

                                                          \
                                           precision_std   
(common, PA) (common, S) (common, support)                 
ALL          201079      0.045                  0.000000   
1.0          185245      0.029                  0.000000   
0.67         15834       0.229                  0.005477   

                                                                           \
                                           fmt_precision_meanstd f1-score   
(common, PA) (common, S) (common, support)                                  
ALL          201079      0.045                   0.66$\pm$0.0000     0.75   
1.0          185245      0.029                   1.00$\pm$0.0000     1.00   
0.67         15834       0.229                   0.37$\pm$0.0055     0.48   

                                                                              \
                                           f1-score_std fmt_f1-score_meanstd   
(common, PA) (common, S) (common, support)                                     
ALL          201079      0.045                      0.0      0.75$\pm$0.0000   
1.0          185245      0.029                      0.0      1.00$\pm$0.0000   
0.67         15834       0.229                      0.0      0.48$\pm$0.0000   

                                                     ... delta_e_ai  Expected  \
                                           accuracy  ...   f1-score precision   
(common, PA) (common, S) (common, support)           ...                        
ALL          201079      0.045                0.970  ...   0.599128  0.659460   
1.0          185245      0.029                1.000  ...   0.890000  1.000000   
0.67         15834       0.229                0.668  ...   0.091833  0.376183   

                                           delta_e_h delta_e_ai  Expected  \
                                           precision  precision    recall   
(common, PA) (common, S) (common, support)                                  
ALL          201079      0.045             -0.000540   0.579460  0.867019   
1.0          185245      0.029              0.000000   0.940000  1.000000   
0.67         15834       0.229              0.002183   0.116183  0.670000   

                                           delta_e_h delta_e_ai  Expected  \
                                              recall     recall  accuracy   
(common, PA) (common, S) (common, support)                                  
ALL          201079      0.045             -0.002981   0.017019  0.974014   
1.0          185245      0.029              0.000000   0.110000  1.000000   
0.67         15834       0.229             -0.002000  -0.120000  0.670000   

                                           delta_e_h delta_e_ai  
                                            accuracy   accuracy  
(common, PA) (common, S) (common, support)                       
ALL          201079      0.045              0.004014   0.414014  
1.0          185245      0.029              0.000000   0.420000  
0.67         15834       0.229              0.002000   0.250000  

[3 rows x 32 columns]

In [8]:


def _prep_display(df_scores):
    display_set_cols = []
    cols_score_types = ['f1-score', "accuracy" ]

    ai_predictor = "AI"
    human_predictor = "another_human"

    for score_type in cols_score_types:
        display_set_cols.append((ai_predictor, score_type))
        display_set_cols.append((human_predictor, f"fmt_{score_type}_meanstd"))
        display_set_cols.append(("Expected", score_type))
        display_set_cols.append(("delta_h_ai", score_type))
        display_set_cols.append(("delta_e_h", score_type))
        display_set_cols.append(("delta_e_ai", score_type))

    return df_scores[display_set_cols].sort_index()

_prep_display(df_scores_all )




,,,AI,another_human,Expected,delta_h_ai,delta_e_h,delta_e_ai,AI,another_human,Expected,delta_h_ai,delta_e_h,delta_e_ai
,,,f1-score,fmt_f1-score_meanstd,f1-score,f1-score,f1-score,f1-score,accuracy,fmt_accuracy_meanstd,accuracy,accuracy,accuracy,accuracy
"(common, PA)","(common, S)","(common, support)",,,,,,,,,,,,
0.67,15834,0.229,0.39,0.48$\pm$0.0000,0.481833,0.09,1.832742e-03,0.091833,0.42,0.67$\pm$0.0045,0.670000,0.248,0.002000,0.250000
1.0,185245,0.029,0.11,1.00$\pm$0.0000,1.000000,0.89,-8.881784e-16,0.890000,0.58,1.00$\pm$0.0000,1.000000,0.420,0.000000,0.420000
ALL,201079,0.045,0.15,0.75$\pm$0.0000,0.749128,0.60,-8.717578e-04,0.599128,0.56,0.97$\pm$0.0000,0.974014,0.410,0.004014,0.414014


In [9]:
print(_prep_display(df_scores_all).reset_index().to_latex(float_format="{:.2f}".format, index=False))

\begin{tabular}{lrlrlrrrrrlrrrr}
\toprule
\multicolumn{3}{r}{common} & AI & another_human & Expected & delta_h_ai & delta_e_h & delta_e_ai & AI & another_human & Expected & delta_h_ai & delta_e_h & delta_e_ai \\
PA & S & support & f1-score & fmt_f1-score_meanstd & f1-score & f1-score & f1-score & f1-score & accuracy & fmt_accuracy_meanstd & accuracy & accuracy & accuracy & accuracy \\
\midrule
0.67 & 15834 & 0.229 & 0.39 & 0.48$\pm$0.0000 & 0.48 & 0.09 & 0.00 & 0.09 & 0.42 & 0.67$\pm$0.0045 & 0.67 & 0.25 & 0.00 & 0.25 \\
1.00 & 185245 & 0.029 & 0.11 & 1.00$\pm$0.0000 & 1.00 & 0.89 & -0.00 & 0.89 & 0.58 & 1.00$\pm$0.0000 & 1.00 & 0.42 & 0.00 & 0.42 \\
ALL & 201079 & 0.045 & 0.15 & 0.75$\pm$0.0000 & 0.75 & 0.60 & -0.00 & 0.60 & 0.56 & 0.97$\pm$0.0000 & 0.97 & 0.41 & 0.00 & 0.41 \\
\bottomrule
\end{tabular}

